# v19 · De Bruijn sequence over the 96 valid codes, and the pointer pathway over 97 symbols

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, PROJ)
mi = load("PtolemyDesktop/Archimedes/Maths/MidlineIndex.py", "mi")
print("VALID_CODES:", len(mi.VALID_CODES), "= K =", mi.K)
r = mi.verify_de_bruijn(2, k=mi.ALPHABET_SIZE); print({k: r[k] for k in ("k", "n", "sequence_length", "all_windows_exactly_once")})
s = mi.de_bruijn_text(2)
wins = Counter(s[i] + s[(i + 1) % len(s)] for i in range(len(s)))
print("96-symbol text, n=2: length", len(s), "| distinct cyclic 2-windows:", len(wins), "| each exactly once:", set(wins.values()) == {1})
assert len(s) == 96**2 and set(wins.values()) == {1}

VALID_CODES: 96 = K = 96
{'k': 96, 'n': 2, 'sequence_length': 9216, 'all_windows_exactly_once': True}
96-symbol text, n=2: length 9216 | distinct cyclic 2-windows: 9216 | each exactly once: True


Every 2-symbol window over the 96 characters occurs exactly once, cyclically, in a string of 96² = 9,216 characters. Moving from one window to the next is a one-symbol shift: this is the traversal order that the `DataStorageNoLocation` branch (v16) uses in place of walking a factorial tree.

## The pointer pathway: first pointer absolute, then deltas (97 symbols)

In [3]:
for w in ["THIS", "hello", "Ptolemy"]:
    p = mi.pathway(w); print(w, p, "->", mi.reconstruct_from_pathway(p)); assert mi.reconstruct_from_pathway(p) == w
allp = [d for w in CORPUS.split() for d in mi.pathway(w)[1:]]
print("deltas over the corpus words: min", min(allp), "max", max(allp), "| distinct:", len(set(allp)), "of a possible", 2*96 + 1)

THIS [45, -12, 1, 10] -> THIS
hello [7, -3, 7, 0, 3] -> hello
Ptolemy [41, -22, -5, -3, -7, 8, 12] -> Ptolemy
deltas over the corpus words: min -86 max 86 | distinct: 130 of a possible 193


The delta between two pointers lies in −96..96, so a delta alone is not bounded by 13 as it is for the letter-from-M step; the bound in the letter encoding comes from the fixed anchor M, not from adjacency.